In [8]:
from google.colab import drive
drive.mount('/content/drive/self_dist')

ValueError: Mountpoint must be in a directory that exists

In [6]:
# prompt: show the files in the current google drive directory

!ls /content/drive/MyDrive


'001508399-evlrpt - Fahad Alwarda - SpanTran (1).pdf'
'001508399-evlrpt - Fahad Alwarda - SpanTran.pdf'
'07.31.2024 Letter to F. Alwarda (1).pdf'
'07.31.2024 Letter to F. Alwarda.pdf'
'Alwarda, Fahad - Resume (1).pdf'
'Alwarda, Fahad - Resume.pdf'
 Carpool.gsheet
'Colab Notebooks'
'Copy of Estate Information Worksheet-Alazzawi, Malak.gdoc'
 CS7643-Assignment2-2.ipynb
 data
 DLA2
 DLA3
 DLA4
'Fahad Alwarda deliverables.zip'
 Fahad_Alwarda_GCS_ATCE
'Fahad Alwarda.pdf'
'Fahad Alward_Resume Extended.pdf'
'Folio 001508399.pdf'
 lecture1.pdf
 lecture1.vtt
'Lilo A. Matrix Review_Qualified.pdf'
 LLM
'Malak+Yamen Applications'
'Market Analysis Fahad Alwarda.pdf'
'My resume'
'my summaries.zip'
'Photo Instruction.pdf'
'Rumaila payslips.HEIC'
'Tax documents '


In [3]:
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoModel
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import os
import json
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------ Crisis Classifier ------------------
class CrisisClassifier(nn.Module):
    def __init__(self, n_classes):
        super(CrisisClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        self.info_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2))
        self.gate = nn.Sequential(
            nn.Linear(768, 768),
            nn.Sigmoid())
        self.crisis_classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, n_classes))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        informativeness_logits = self.info_classifier(pooled_output)
        informativeness_probs = torch.softmax(informativeness_logits, dim=1)
        informativeness_score = informativeness_probs[:, 1].unsqueeze(1)
        gated_output = informativeness_score * pooled_output
        crisis_logits = self.crisis_classifier(gated_output)
        return informativeness_logits, crisis_logits

# ------------------ Dataset ------------------
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


In [5]:
val_df = pd.read_csv("combined_dev.tsv", sep="\t").dropna().reset_index(drop=True)


FileNotFoundError: [Errno 2] No such file or directory: 'combined_dev.tsv'

In [8]:

class CrisisDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.max_len = max_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        encoding = self.tokenizer(
            row['text'],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'crisis_type': int(row["humanitarian_label"]),
            'informativeness': int(row["informativeness_label"]),
        }

# ------------------ Train Function ------------------
def train(model, dataloader, optimizer, criterion_informativeness, criterion_crisis, device, teacher_logits=None, alpha=0.5):
    model.train()
    total_loss = 0

    for idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        crisis_labels = batch['crisis_type'].to(device)
        informativeness_labels = batch['informativeness'].to(device)

        optimizer.zero_grad()
        informativeness_logits, crisis_logits = model(input_ids, attention_mask)

        loss1 = criterion_informativeness(informativeness_logits, informativeness_labels)

        informative_mask = informativeness_labels == 1
        if informative_mask.sum() > 0:
            loss2 = criterion_crisis(crisis_logits[informative_mask], crisis_labels[informative_mask])
        else:
            loss2 = torch.tensor(0.0, device=device)

        loss = loss1 + loss2

        # 🔥 Self-distillation loss if teacher logits are provided
        if teacher_logits is not None:
            with torch.no_grad():
                t_info_logits, t_crisis_logits = teacher_logits[idx]
            distill_loss_info = F.kl_div(
                F.log_softmax(informativeness_logits / 1.0, dim=1),
                F.softmax(t_info_logits / 1.0, dim=1),
                reduction='batchmean'
            )
            distill_loss_crisis = F.kl_div(
                F.log_softmax(crisis_logits / 1.0, dim=1),
                F.softmax(t_crisis_logits / 1.0, dim=1),
                reduction='batchmean'
            )
            distill_loss = distill_loss_info + distill_loss_crisis
            loss = alpha * loss + (1 - alpha) * distill_loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

# ------------------ Evaluation Function ------------------
def eval_model(model, dataloader, device):
    model.eval()
    all_info_preds, all_info_labels = [], []
    all_crisis_preds, all_crisis_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            info_true = batch["informativeness"].to(device)
            crisis_true = batch["crisis_type"].to(device)

            informativeness_logits, crisis_logits = model(input_ids, attention_mask)

            info_preds = torch.argmax(informativeness_logits, dim=1).cpu().numpy()
            crisis_preds = torch.argmax(crisis_logits, dim=1).cpu().numpy()

            all_info_preds.extend(info_preds)
            all_info_labels.extend(batch["informativeness"].cpu().numpy())
            all_crisis_preds.extend(crisis_preds)
            all_crisis_labels.extend(batch["crisis_type"].cpu().numpy())

    return np.array(all_info_labels), np.array(all_info_preds), np.array(all_crisis_labels), np.array(all_crisis_preds)

# ------------------ Plot Confusion Matrix ------------------
def plot_cm(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title(title)
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=300)
    plt.close()

# ------------------ Load Data ------------------
train_df = pd.read_csv("/content/combined_dev.csv", sep="\t").dropna().reset_index(drop=True)
val_df = pd.read_csv("/content/combined_dev.tsv", sep="\t").dropna().reset_index(drop=True)

info_labels = ['not_informative', 'informative']
info2id = {label: idx for idx, label in enumerate(info_labels)}
id2info = {idx: label for label, idx in info2id.items()}
hum_labels = sorted(train_df["humanitarian_label"].unique())
hum2id = {label: idx for idx, label in enumerate(hum_labels)}
id2hum = {idx: label for label, idx in hum2id.items()}

train_df["informativeness_label"] = train_df["informativeness_label"].map(info2id)
val_df["informativeness_label"] = val_df["informativeness_label"].map(info2id)
train_df["humanitarian_label"] = train_df["humanitarian_label"].map(hum2id)
val_df["humanitarian_label"] = val_df["humanitarian_label"].map(hum2id)

data_train = CrisisDataset(train_df, tokenizer, max_len=128)
data_val = CrisisDataset(val_df, tokenizer, max_len=128)

train_loader = DataLoader(data_train, batch_size=32, shuffle=True)
val_loader = DataLoader(data_val, batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

# ------------------ Step 1: Train Teacher Model ------------------
model_teacher = CrisisClassifier(n_classes=len(hum_labels)).to(device)
optimizer = torch.optim.AdamW(model_teacher.parameters(), lr=2e-5)
criterion_informativeness = nn.CrossEntropyLoss()
criterion_crisis = nn.CrossEntropyLoss()

print("\n🛠️ Training Teacher Model...")
for epoch in range(3):
    train_loss = train(model_teacher, train_loader, optimizer, criterion_informativeness, criterion_crisis, device)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

# Evaluate teacher
info_true, info_preds, crisis_true, crisis_preds = eval_model(model_teacher, val_loader, device)

print("\n📜 Classification Report (Teacher)")
print(classification_report(info_true, info_preds, target_names=[id2info[i] for i in sorted(id2info)]))
print(classification_report(crisis_true, crisis_preds, target_names=[id2hum[i] for i in sorted(id2hum)]))
plot_cm(info_true, info_preds, [id2info[i] for i in sorted(id2info)], "Teacher_Informativeness")
plot_cm(crisis_true, crisis_preds, [id2hum[i] for i in sorted(id2hum)], "Teacher_Crisis")

# Save teacher logits
teacher_logits = []
with torch.no_grad():
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        info_logits, crisis_logits = model_teacher(input_ids, attention_mask)
        teacher_logits.append((info_logits.detach(), crisis_logits.detach()))

# ------------------ Step 2: Train Student Model with Self-Distillation ------------------
model_student = CrisisClassifier(n_classes=len(hum_labels)).to(device)
optimizer = torch.optim.AdamW(model_student.parameters(), lr=2e-5)

print("\n🛠️ Training Student Model (Self-Distillation)...")
for epoch in range(3):
    train_loss = train(model_student, train_loader, optimizer, criterion_informativeness, criterion_crisis, device, teacher_logits, alpha=0.5)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")

# Evaluate student
info_true, info_preds, crisis_true, crisis_preds = eval_model(model_student, val_loader, device)

print("\n📜 Classification Report (Student)")
print(classification_report(info_true, info_preds, target_names=[id2info[i] for i in sorted(id2info)]))
print(classification_report(crisis_true, crisis_preds, target_names=[id2hum[i] for i in sorted(id2hum)]))
plot_cm(info_true, info_preds, [id2info[i] for i in sorted(id2info)], "Student_Informativeness")
plot_cm(crisis_true, crisis_preds, [id2hum[i] for i in sorted(id2hum)], "Student_Crisis")


FileNotFoundError: [Errno 2] No such file or directory: '/content/combined_dev.tsv'